# S3 Object Storage Uploads

MinIO/S3-compatible object listing, file manifests and recursive image uploads.

Stage: prototype.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
! pip install minio

In [ ]:
import os
from minio import Minio
from minio.error import S3Error

bucket_name = os.environ["MINIO_BUCKET"]
eos_client = Minio(
    os.environ["MINIO_ENDPOINT"],
    access_key=os.environ["MINIO_ACCESS_KEY"],
    secret_key=os.environ["MINIO_SECRET_KEY"],
    secure=True,
)


In [ ]:
# method: list_objects
# params: bucket_name, prefix (object path prefix), recursive (set True for directories)

objects = eos_client.list_objects(bucket_name,
                                  recursive=True)

list_obj = []
for obj in objects:
  if 'en/' in obj.object_name:
    print(obj.bucket_name, obj.object_name, obj.last_modified, obj.size, obj.content_type)
    list_obj.append(obj.object_name)

In [ ]:
len(list_obj)

In [ ]:

# Specify the file path
file_path = "output_en.txt"

# Open the file in write mode and write each string in a new line
with open(file_path, 'w') as file:
    for item in list_obj:
        file.write("%s\n" % item)

print(f"The list has been saved to {file_path}")

In [ ]:
'https://objectstore.e2enetworks.net/indic-ocr/' + filename

In [ ]:
# # List of numbers of Bengali
# numbers = [9536, 11677, 2140, 1210, 927, 456, 330, 1021, 464, 1054, 831, 841]

# # Truncate each number to not more than 1500
# truncated_numbers = [min(num, 1500) for num in numbers]
# print(truncated_numbers)
# # Add up the truncated numbers
# sum_of_truncated_numbers = sum(truncated_numbers)

# print(f"The sum of truncated numbers is: {sum_of_truncated_numbers}")

In [ ]:


main_folder_path = os.environ["UPLOAD_SOURCE_DIR"]
minio_prefix = os.environ.get("UPLOAD_PREFIX", "images")

# Ensure the destination folder exists on Minio
if not eos_client.bucket_exists(bucket_name):
    eos_client.make_bucket(bucket_name)

for root, _, files in os.walk(main_folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        relative_path = os.path.relpath(file_path, main_folder_path)

        try:
            with open(file_path, 'rb') as file_data:
                file_stat = os.stat(file_path)
                length = file_stat.st_size

                remote_file_path = os.path.join(minio_prefix, relative_path)

                content_type = 'image/png' if file_path.lower().endswith('.png') else 'image/jpeg'

                eos_client.put_object(bucket_name, remote_file_path, file_data, length, content_type=content_type)
                print(f"Uploaded: {file_path}")

        except Exception as e:
            print(f"Error uploading {file_path}: {e}")


In [ ]:
if os.environ.get("ALLOW_OBJECT_DELETION") == "YES":
    # method: remove_objects
    # params: bucket_name, object prefix (e.g. directory path)
    from minio.error import S3Error

    try:
        get_name = lambda object: object.object_name
        names = map(get_name, eos_client.list_objects(bucket_name, 'indic-ocr/', recursive=True))
        print(list(names))
        for obj in list(names):
          eos_client.remove_object(bucket_name, obj)
        # for err in eos_client.remove_objects('bucketname', names):
        #     print("Deletion Error: {}".format(err))
    except S3Error as err:
        print(err)

In [ ]:
if os.environ.get("ALLOW_OBJECT_DELETION") == "YES":
    # method: remove_object
    # params: bucket_name, object name (object full path)
    from minio.error import S3Error

    try:
        eos_client.remove_object(bucket_name, 'indic-ocr/')
    except ResponseError as err:
        print(err)